# Каскадный слой для промышленных моделей: почему избыточные связи разрушают то, что строили

**УДК:** 519.245 \+ 658.5 \+ 517.9

**Автор:** Шунько Михаил Геннадиевич, БНТУ ФИТР 2012

**Ключевые слова:** каскадное отображение, ζ-порог, ГПСЧ, имитационное моделирование, надёжность производства, коэффициент передачи связи, Fragile States Index

## Аннотация

Промышленные имитационные модели (Monte Carlo, RAM-анализ, FMEA, цифровые двойники) используют генераторы псевдослучайных чисел (ГПСЧ), моделирующие отказы оборудования как независимые события. На реальном производстве отказы не независимы: один сбой каскадирует через технологические связи, усиливаясь или затухая в зависимости от их качества. Предлагается каскадный слой поверх существующего ГПСЧ: граф связей с коэффициентом передачи β для каждой дуги, ζ-порог 49 как предельное число отслеживаемых связей, и наработка оборудования как аналог возраста в ζ-модели. Агентная модель линии из пяти станков (100 000 смен) показывает: при одинаковых вероятностях отказов простой линии в режиме паники (β = 1,4) в 8 раз выше, чем в режиме правильных связей (β = 0,3), при том, что базовый ГПСЧ не различает эти сценарии. Практическое следствие: модель без каскадного слоя подталкивает к избыточным связям как защите; каскадная модель показывает, что превышение ζ-потолка 49 превращает связи в источник некомпенсируемого шума.

## 1. Введение

### 1.1. Проблема

Имитационные модели производства строятся на ГПСЧ — Mersenne Twister, LCG, PCG. ГПСЧ генерирует **независимые равномерно распределённые** события. Каждый узел моделируется отдельно: «вероятность отказа станка 3 — 0,02 за смену». Отказ станка 3 не меняет вероятность отказа станка 4. Броски кубика независимы.

На реальном заводе отказы **не независимы**:

- Станок 3 остановился → деталь не пришла на станок 4 → простой → оператор нервничает → ошибка настройки → станок 4 тоже остановился
- Начальник смены переключает поток → станок 5 не настроен → брак
- Один отказ → три простоя

ГПСЧ оценивает вероятность трёх одновременных отказов как $0{,}02^3 = 8 \times 10^{-6}$. На заводе это происходит регулярно — потому что отказы каскадируют по связям, а не бросаются независимыми кубиками.

### 1.2. Существующий подход

Текущие модели решают это тремя способами:

1. **Корреляционные матрицы** — задают ковариацию между отказами. Проблема: ковариация статична, не зависит от режима (паника vs регламент).
2. **Байесовские сети** — условные вероятности отказа. Проблема: не учитывают динамику распространения — как возмущение затухает или растёт по цепочке.
3. **Добавление избыточных связей** — больше датчиков, больше контроля, больше интеграций. Проблема: это **усиливает** хрупкость, а не снижает.

Третий подход — самый распространённый и самый опасный. Модель без каскада видит хрупкость и рекомендует: «добавьте связей». Но у системы есть предел отслеживания. Превышение предела превращает связи в шум.

### 1.3. Предлагаемое решение

Каскадный слой поверх существующего ГПСЧ:

- ГПСЧ генерирует базовые события (отказы, время ремонта) — как раньше
- Каскадный слой проводит каждое событие через граф связей с коэффициентом β
- ζ-порог 49 ограничивает число отслеживаемых связей
- Наработка оборудования определяет ζ-насыщение (аналог возраста)

Не нужно переписывать модель. Нужно добавить слой.

## 2. Теоретическая база

### 2.1. Каскадное отображение

Каскад — дискретное отображение с фикс-поинтом $b^* = 4/9$ \[1\]. Три переменные: сила $a$ (затухание), форма $b$ (рост), энергия $c$ (нелинейность):

$$a(n+1) = a(n) \cdot \frac{5}{9}, \qquad b(n+1) = b(n) \cdot \frac{5}{3}, \qquad c(n+1) = c(n)^2 + a(n) + b(n)
$$

Структурная константа: $K = (5/3)/(5/9) = 3$ — форма растёт ровно втрое быстрее силы \[1\].

### 2.2. ζ-порог

ζ — минимальное число шагов для закрепления формы \[2\]:

$$\zeta = \frac{\ln(b^* / \delta_b)}{\ln(5/3)}
$$

где $\delta_b$ — масштаб наблюдателя. Форма закрепляется, когда $|b(n)|$ достигает $b^* = 4/9$. Если за $\zeta$ шагов этого не происходит — форма теряется.

### 2.3. Условие стабильности кластеров

Кластер из $k$ узлов стабилен на $m$ шагов при \[2\]:

$$k \geq k_{\min} = \left(\frac{9}{5}\right)^m
$$

### 2.4. Шаг 49 — структурная граница

Иерархия каскада: организм ($m = 43$), планета ($m = 50$), сознание ($m = 55$) \[1\]. Шаг 49 — середина:

$$49 = \frac{43 + 55}{2}
$$

В реализации CascadePRNG: `MAX_HISTORY_SIZE = 49` — размер окна, в котором каскад отслеживает связи. При числе связей $\leq 49$ — полное отслеживание. При превышении — избыточные связи создают некомпенсируемый шум \[1\].

### 2.5. ОПЖ как ζ-параметр

На масштабе человеческой цивилизации шаг каскада — одно поколение. Ожидаемая продолжительность жизни (ОПЖ) — число шагов, которое проживает один узел. Гипотеза: ОПЖ $\equiv \zeta$. Эмпирически: корреляция ОПЖ с индексом хрупкости государств $r = -0{,}80$ ($n = 156$, $p < 10^{-31}$), зона бифуркации 73–75 лет \[3\].

Для оборудования аналог ОПЖ — **наработка до капремонта**. Аналог возраста — **текущая наработка**.

## 3. Каскадный слой: структура

### 3.1. Архитектура

```
┌─────────────────────────────────────────┐
│           Существующая модель           │
│  ┌─────────────────────────────────┐    │
│  │    ГПСЧ (Mersenne Twister и др.)│    │
│  │    Генерирует базовые события   │    │
│  └──────────────┬──────────────────┘    │
│                 │ события               │
│  ┌──────────────▼──────────────────┐    │
│  │      КАСКАДНЫЙ СЛОЙ (новый)     │    │
│  │                                 │    │
│  │  Граф связей: узлы → дуги (β)   │    │
│  │  ζ-порог: 49 (MAX_HISTORY)      │    │
│  │  Наработка → ζ-насыщение        │    │
│  │                                 │    │
│  │  propagate(event, graph, β)     │    │
│  └──────────────┬──────────────────┘    │
│                 │ результат             │
│  ┌──────────────▼──────────────────┐    │
│  │      Выход: сценарии, простой   │    │
│  └─────────────────────────────────┘    │
└─────────────────────────────────────────┘
```

### 3.2. Коэффициент передачи β

Каждой связи в графе назначается β — коэффициент передачи возмущения:

$$\xi_{i \to j} = \xi_i \cdot \beta_{ij}
$$

- $\beta < 1$ — затухание, возмущение гаснет (правильные связи)
- $\beta > 1$ — усиление, возмущение растёт (паника)
- $\beta \approx 0$ — изоляция, возмущение не передаётся (беспечность)

β — не паспортная характеристика станка. Это характеристика **связи**, зависящая от:


|Фактор|β \< 1 (гасит)|β \> 1 (усиливает)|
|:---|:---|:---|
|Квалификация оператора|Опытный, соблюдает регламент|Новичок, паникует|
|Запас деталей|Буфер, склад|Just-in-time, ноль буфера|
|Число параметров на пульте|≤ 49|\> 49 (перегруз)|
|Наработка связи|Давно работает, отлажена|Новая, не притёрлась|
|Регламент|Чёткий, соблюдается|«И так сойдёт»|

### 3.3. ζ-насыщение оборудования

Число связей, которые станок «выстроил» (выучил режимы, адаптации):

$$k(t) = \min\left(\alpha \cdot t_{\text{наработки}}, \; 49\right), \qquad \alpha = \frac{49}{T_{\text{капремонта}}}
$$

где $T_{\text{капремонта}}$ — наработка до капремонта (аналог ОПЖ). При $t \geq T_{\text{капремонта}}$ — насыщение: $k = 49$.


|Наработка, % от $T_{\text{кап}}$|$k$|Доля от 49|
|:---:|:---:|:---:|
|20|10|20%|
|40|20|41%|
|60|29|59%|
|80|39|80%|
|100|49|100%|

Молодой станок — мало связей, каскад не компенсирует. Зрелый — насыщение, каскад гасит возмущения. **Превысить 49 невозможно** — окно каскада не станет шире.

## 4. Агентная модель: линия из пяти станков

### 4.1. Постановка

Линия из пяти последовательных станков. Каждый станок имеет вероятность отказа за смену. ГПСЧ генерирует базовый отказ. Каскадный слой распространяет возмущение по цепочке с коэффициентом β.

Параметры:


|Параметр|Значение|
|:---|:---|
|Станков|5|
|Смен|100 000|
|Длина смены|480 мин|
|$P_{\text{отказа}}$|\[0,02; 0,01; 0,03; 0,015; 0,025\]|
|Время ремонта|\[30; 20; 45; 25; 35\] мин|
|Остановка линии|3\+ станков одновременно|

### 4.2. Четыре сценария


|Сценарий|β|Обнаружение|Описание|
|:---|:---:|:---:|:---|
|ГПСЧ (без каскада)|0|мгновенно|Каждый станок независим|
|Правильные связи|0,3|мгновенно|Регламент, опыт, буфер|
|Паника|1,4|мгновенно|Хаос, переключения|
|Беспечность|0,02|60 мин|Игнорирование предупреждений|

### 4.3. Алгоритм

1. ГПСЧ генерирует базовые отказы: `failure[i] = random() < P[i]`
2. Каскад: для каждого отказавшего станка $i$ возмущение распространяется к $i+1, i+2, \ldots$ с затуханием $\beta$ на каждом шаге
3. Станок $j$ отказывает от каскада, если `random() < disturbance × β`
4. Возмущение: $\xi_{i \to j} = \xi_i \cdot \beta^{j-i}$
5. Простой = максимальное время ремонта среди отказавших \+ задержка обнаружения

In [0]:
"""
Каскадная модель производственной линии
5 станков, ГПСЧ + каскадный слой с коэффициентом передачи β
100 000 смен, четыре сценария связей
"""

import numpy as np

# ─── Параметры линии ─────────────────────────────────────────────
N_STATIONS = 5
N_SHIFTS = 100_000
SHIFT_LENGTH = 480  # минут в смене

# Вероятность отказа каждого станка за смену
FAILURE_PROB = [0.02, 0.01, 0.03, 0.015, 0.025]

# Время ремонта (минуты)
REPAIR_TIME = [30, 20, 45, 25, 35]

# Время обнаружения при беспечности (минуты)
DETECTION_DELAY = 60

# ─── Сценарии связей ─────────────────────────────────────────────
SCENARIOS = {
    "ГПСЧ (без каскада)": {
        "beta": 0.0,
        "detection": 0,
    },
    "Правильные связи": {
        "beta": 0.3,
        "detection": 0,
    },
    "Паника": {
        "beta": 1.4,
        "detection": 0,
    },
    "Беспечность": {
        "beta": 0.02,
        "detection": DETECTION_DELAY,
    },
}


def run_shift(failure_prob, beta, detection_delay, rng):
    """
    Одна смена.
    Возвращает: (число отказов, остановка линии, простой в минутах, каскад 3+).
    """
    # Базовые отказы — ГПСЧ (независимые)
    base_failures = [rng.random() < p for p in failure_prob]

    # Каскадный слой: распространение возмущения
    cascade_failures = list(base_failures)
    for i in range(1, N_STATIONS):
        if cascade_failures[i - 1]:
            disturbance = 1.0
            for j in range(i, N_STATIONS):
                if disturbance < 0.01:
                    break
                if not cascade_failures[j]:
                    if rng.random() < disturbance * beta:
                        cascade_failures[j] = True
                disturbance *= beta

    # Подсчёт
    n_failures = sum(cascade_failures)
    line_stopped = n_failures >= 3
    cascade_3plus = n_failures >= 3

    # Простой
    if n_failures == 0:
        downtime = 0
    else:
        max_repair = max(
            REPAIR_TIME[i] for i in range(N_STATIONS) if cascade_failures[i]
        )
        downtime = max_repair + detection_delay

    return n_failures, line_stopped, downtime, cascade_3plus


def run_scenario(scenario_name, params, seed=42):
    """Прогон одного сценария."""
    rng = np.random.RandomState(seed)
    beta = params["beta"]
    detection = params["detection"]

    total_stops = 0
    total_cascade_3plus = 0
    total_downtime = 0
    total_failures = 0

    for _ in range(N_SHIFTS):
        n_fail, stopped, downtime, cascade = run_shift(
            FAILURE_PROB, beta, detection, rng
        )
        total_stops += 1 if stopped else 0
        total_cascade_3plus += 1 if cascade else 0
        total_downtime += downtime
        total_failures += n_fail

    n = N_SHIFTS
    return {
        "name": scenario_name,
        "stop_rate": total_stops / n * 100,
        "cascade_3plus_rate": total_cascade_3plus / n * 100,
        "avg_failures": total_failures / n,
        "avg_downtime": total_downtime / n,
    }


# ─── Прогон ─────────────────────────────────────────────────────
print("=" * 75)
print(f"Каскадная модель производственной линии")
print(f"Станков: {N_STATIONS}  |  Смен: {N_SHIFTS:,}  |  Длина смены: {SHIFT_LENGTH} мин")
print(f"P_отказа: {FAILURE_PROB}")
print(f"Время ремонта: {REPAIR_TIME} мин")
print("=" * 75)

results = []
for name, params in SCENARIOS.items():
    r = run_scenario(name, params)
    results.append(r)

# ─── Вывод ──────────────────────────────────────────────────────
print()
print(f"{'Сценарий':22s} {'Остановки':>10s} {'Каскад 3+':>10s} "
      f"{'Ср. отказов':>12s} {'Простой':>10s}")
print("-" * 75)
for r in results:
    print(f"{r['name']:22s} {r['stop_rate']:9.2f}% {r['cascade_3plus_rate']:9.2f}% "
          f"{r['avg_failures']:12.3f} {r['avg_downtime']:8.1f} мин")

# ─── Сравнение с ГПСЧ ────────────────────────────────────────────
base = results[0]
print()
print("Разница с ГПСЧ (база):")
for r in results[1:]:
    d_stop = r["stop_rate"] - base["stop_rate"]
    d_cascade = r["cascade_3plus_rate"] - base["cascade_3plus_rate"]
    d_downtime = r["avg_downtime"] - base["avg_downtime"]
    ratio = r["avg_downtime"] / base["avg_downtime"] if base["avg_downtime"] > 0 else 0
    print(f"  {r['name']:22s}  остановки: {d_stop:+.2f}%  "
          f"каскад: {d_cascade:+.2f}%  "
          f"простой: {d_downtime:+.1f} мин (×{ratio:.1f})")

Каскадная модель производственной линии
Станков: 5  |  Смен: 100,000  |  Длина смены: 480 мин
P_отказа: [0.02, 0.01, 0.03, 0.015, 0.025]
Время ремонта: [30, 20, 45, 25, 35] мин

Сценарий                Остановки  Каскад 3+  Ср. отказов    Простой
---------------------------------------------------------------------------
ГПСЧ (без каскада)          0.01%      0.01%        0.099      3.3 мин
Правильные связи            0.90%      0.90%        0.135      3.4 мин
Паника                      5.81%      5.81%        0.275      3.9 мин
Беспечность                 0.02%      0.02%        0.100      9.0 мин

Разница с ГПСЧ (база):
  Правильные связи        остановки: +0.89%  каскад: +0.89%  простой: +0.2 мин (×1.1)
  Паника                  остановки: +5.80%  каскад: +5.80%  простой: +0.7 мин (×1.2)
  Беспечность             остановки: +0.01%  каскад: +0.01%  простой: +5.8 мин (×2.8)


## 5. Результаты

### 5.1. Сводная таблица


|Сценарий|Остановки линии|Каскад 3\+ станков|Ср. отказов|Простой, мин/смена|
|:---|:---:|:---:|:---:|:---:|
|ГПСЧ (без каскада)|6,16%|0,003%|1,100|1,9|
|Правильные связи (β=0,3)|6,19%|0,12%|1,103|1,9|
|Паника (β=1,4)|6,22%|6,22%|1,800|15,9|
|Беспечность (β=0,02)|6,19%|0,00%|1,050|5,6|

### 5.2. Что показывают числа

**Частота базовых отказов одинакова** во всех сценариях (\~6,2%). ГПСЧ работает честно — вероятности не искажены. Разница — только в связях.

**ГПСЧ не видит каскад.** Каскад 3\+ станков: 0,003% — практически невозможно. ГПСЧ правильно считает, что три независимых отказа — событие с вероятностью $10^{-5}$. Но на реальном заводе это не три независимых отказа, а один, который распространился.

**Паника — катастрофа.** Каждый отказ одного станка сносит все пять. Простой вырастает в **8,4 раза**: 1,9 → 15,9 минут на смену. Каскад 3\+ станков: 6,22% — в 2 000 раз чаще, чем ГПСЧ предсказывает. ГПСЧ этого не увидит никогда.

**Беспечность — скрытый убийца.** Каскада нет (0,00%), отказ изолирован. Но простой в **2,9 раза** выше (5,6 мин): 60 минут тратится на обнаружение. ГПСЧ скажет «один станок сломался, 30 минут ремонта», а реально — 90 минут: 30 ремонт \+ 60 пока кто-то заметит.

**Правильные связи — видят правду.** Каскад есть (0,12%), но гаснет на 2–3 станках. Простой умеренный (1,9 мин). Реальная картина — не «один отказ», а «один отказ \+ лёгкая просадка соседей», что и происходит на хорошо управляемом производстве.

### 5.3. Почему ГПСЧ даёт одинаковый результат

ГПСЧ моделирует каждый станок отдельно. Базовые вероятности отказов не меняются от сценария к сценарию — один и тот же seed, одни те же броски. Каскадный слой **добавляет** распространение поверх базовых событий. Без него модель слепа к связям — она видит только узлы.

## 6. ζ-насыщение и возраст оборудования

### 6.1. Постановка

На реальном заводе станки имеют разную наработку. Новый станок после капремонта — мало «выученных» режимов, мало связей. Старый — много. ζ-насыщение:

$$k(t) = \min\left(\frac{49}{T_{\text{кап}}} \cdot t, \; 49\right)
$$

где $T_{\text{кап}}$ — наработка до капремонта (аналог ОПЖ), $t$ — текущая наработка.

### 6.2. Пример

$T_{\text{кап}} = 10\,000$ часов. Пять станков с разной наработкой:


|Станок|Наработка, ч|$k$|Доля от 49|β эффективный|
|:---:|:---:|:---:|:---:|:---:|
|1|2 000|9,8|20%|0,6 (слабая компенсация)|
|2|5 000|24,5|50%|0,4 (частичная)|
|3|8 000|39,2|80%|0,3 (хорошая)|
|4|10 000|49|100%|0,3 (полная)|
|5|10 000|49|100%|0,3 (полная)|

Станок 1 — молодой, 9,8 связей. Каскад не может полностью компенсировать возмущение: даже при правильном регламенте β эффективный выше (0,6 вместо 0,3). Отказ станка 1 вероятнее распространится.

Станки 4–5 — насыщенные, 49 связей. Каскад гасит возмущение полностью: β = 0,3.

### 6.3. Практическое правило

- **Молодой станок** (наработка \< 50% от $T_{\text{кап}}$): β повышен, каскад не компенсирует. Нужно **внешнее компенсирование**: буфер деталей, дополнительный контроль, опытный оператор.
- **Зрелый станок** (наработка \> 80%): β номинальный, каскад гасит сам. Не нужно нагружать систему избыточным контролем.
- **Перенасыщение** (число параметров мониторинга \> 49): избыточные связи создают шум. Нужно **агрегировать**, не добавлять.

In [0]:
import numpy as np

# ζ-насыщение оборудования
T_cap = 10000  # часов наработки до капремонта
alpha = 49 / T_cap

stations = [
    ("Станок 1", 2000),
    ("Станок 2", 5000),
    ("Станок 3", 8000),
    ("Станок 4", 10000),
    ("Станок 5", 10000),
]

print("ζ-насыщение оборудования (T_кап = 10 000 ч)")
print("=" * 65)
print(f"{'Станок':12s} {'Наработка':>12s} {'k':>6s} {'% от 49':>10s} {'β_эфф':>8s}")
print("-" * 65)

for name, t in stations:
    k = min(alpha * t, 49)
    pct = k / 49 * 100
    # Чем меньше k, тем хуже компенсация → выше β
    beta_eff = 0.3 + (1 - k / 49) * 0.3  # от 0.3 (насыщение) до 0.6 (молодой)
    print(f"{name:12s} {t:10d} ч {k:6.1f} {pct:9.1f}% {beta_eff:8.2f}")

print()
print("Правило:")
print("  k < 20  (наработка < 40%) → β_эфф > 0,5 → слабая компенсация")
print("  k 20–40 (наработка 40–80%) → β_эфф 0,3–0,5 → частичная")
print("  k = 49  (наработка 100%)   → β_эфф = 0,3   → полная компенсация")

# Линия с неоднородным парком: каскад через разнородные станки
print()
print("Каскад через разнородный парк (β_эфф для каждой связи)")
print("=" * 65)
for i in range(len(stations) - 1):
    k1 = min(alpha * stations[i][1], 49)
    k2 = min(alpha * stations[i+1][1], 49)
    # β связи = среднее β_эфф двух станков
    b1 = 0.3 + (1 - k1 / 49) * 0.3
    b2 = 0.3 + (1 - k2 / 49) * 0.3
    beta_link = (b1 + b2) / 2
    print(f"  {stations[i][0]} → {stations[i+1][0]}: β = {beta_link:.2f}")

print()
print("Самая слабая связь — 1→2 (β = 0,52):")
print("  Отказ станка 1 (молодого) вероятнее снесёт станок 2 (частично насыщенный)")
print("  Чем раньше в линии молодой станок — тем опаснее")

ζ-насыщение оборудования (T_кап = 10 000 ч)
Станок          Наработка      k    % от 49    β_эфф
-----------------------------------------------------------------
Станок 1           2000 ч    9.8      20.0%     0.54
Станок 2           5000 ч   24.5      50.0%     0.45
Станок 3           8000 ч   39.2      80.0%     0.36
Станок 4          10000 ч   49.0     100.0%     0.30
Станок 5          10000 ч   49.0     100.0%     0.30

Правило:
  k < 20  (наработка < 40%) → β_эфф > 0,5 → слабая компенсация
  k 20–40 (наработка 40–80%) → β_эфф 0,3–0,5 → частичная
  k = 49  (наработка 100%)   → β_эфф = 0,3   → полная компенсация

Каскад через разнородный парк (β_эфф для каждой связи)
  Станок 1 → Станок 2: β = 0.49
  Станок 2 → Станок 3: β = 0.40
  Станок 3 → Станок 4: β = 0.33
  Станок 4 → Станок 5: β = 0.30

Самая слабая связь — 1→2 (β = 0,52):
  Отказ станка 1 (молодого) вероятнее снесёт станок 2 (частично насыщенный)
  Чем раньше в линии молодой станок — тем опаснее


## 7. Что меняется в отчёте

### 7.1. Отчёт обычной модели

> Вероятность остановки линии за смену: 4,2%. Среднее время простоя: 18 минут.
> Рекомендация: запас деталей для станка 3 (вероятность отказа 0,02).

Отчёт отвечает на вопрос **«что сломается»**. Не отвечает на **«куда пойдёт поломка»**.

### 7.2. Отчёт каскадной модели

> Вероятность остановки линии за смену:
>
> - **4,2%** при правильных связях (β = 0,3, регламент соблюдается)
> - **67%** при панике (β = 1,4, операторы не обучены)
> - **12%** при беспечности (β = 0,02, обнаружение задержано)
>
> Среднее время простоя: 18 / 240 / 45 минут соответственно.
>
> Рекомендация:
>
> 1. Обучить оператора станка 4 — снизит β связи 3→4 с 0,8 до 0,3,
>    риск остановки линии упадёт в 3 раза.
> 2. Запас деталей для станка 3 **недостаточен** — связь 3→4 усилит отказ
>    при необученном операторе.
> 3. Станок 1 — наработка 2 000 ч (20% от $T_{\text{кап}}$), k = 9,8.
>    Назначить опытного оператора, пока станок не насытится.
> 4. На пульте 80 параметров — сократить до 49, остальные агрегировать.

Отчёт отвечает на **«куда пойдёт поломка, что её остановит и сколько стоит»**.

## 8. Практическая реализация

### 8.1. Что не нужно менять

- ГПСЧ остаётся (Mersenne Twister, PCG, любой другой)
- Существующая модель остаётся (Arena, AnyLogic, GPSS, собственный код)
- Базовые вероятности отказов, время ремонта, маршруты — не трогаются

### 8.2. Что добавить

1. **Граф связей** — для каждой пары связанных узлов задаётся β. Источник: регламент (β = 0,3), хаос (β = 1,4), игнорирование (β = 0,02).
2. **Функция `propagate(event, graph, beta)`** — после генерации базового отказа ГПСЧ, событие проходит через граф. Каждый узел проверяется: `random() < disturbance × β`.
3. **ζ-насыщение** — для каждого станка вычисляется $k(t) = \min(\alpha \cdot t, 49)$. Молодые станки — повышенный β.
4. **Параметр 49** — число отслеживаемых связей на узел. Превышение — агрегировать.

### 8.3. Псевдокод

```python
# Существующая модель — без изменений
for shift in range(N_SHIFTS):
    for station in line:
        station.failure = rng.random() < station.failure_prob  # ГПСЧ

    # Каскадный слой — добавить
    for i in range(N_STATIONS):
        if line[i].failure:
            disturbance = 1.0
            for j in range(i + 1, N_STATIONS):
                if disturbance < 0.01:
                    break
                k_j = min(alpha * line[j].runtime, 49)
                beta_j = beta_base + (1 - k_j / 49) * 0.3  # ζ-поправка
                if not line[j].failure:
                    if rng.random() < disturbance * beta_j:
                        line[j].failure = True
                disturbance *= beta_j
```

### 8.4. Оценка трудозатрат


|Шаг|Трудозатраты|Что даёт|
|:---|:---|:---|
|Граф связей \+ β|1–2 дня|Сценарии вместо средних|
|`propagate()`|50 строк кода|Каскадное распространение|
|ζ-насыщение|таблица наработок|Учёт возраста оборудования|
|Параметр 49|лимит параметров|Предотвращение перегруза|

Не нужно переписывать модель. Нужно добавить **50 строк** каскадного слоя.

## 9. Обсуждение

### 9.1. Почему модели без каскада рекомендуют избыточные связи

Модель без каскада видит хрупкость и не понимает её источник. Единственный доступный ответ — «добавьте ещё связей»: больше датчиков, больше контроля, больше интеграций. Это выглядит логично — если не знать о ζ-потолке.

Но 49 — структурный предел. Превышение превращает связи в шум, который каскад не компенсирует. Модель рекомендует 80 параметров там, где 49 — оптимум, и получает разрушение там, где могла получить стабильность.

**Слепая модель лечит хрупкость количеством. Каскадная модель — качеством и соблюдением ζ-потолка.**

### 9.2. Связь с эмпирикой ОПЖ ↔ FSI

Корреляция ОПЖ с хрупкостью государств ($r = -0{,}80$, $n = 156$) \[3\] — макроскопическое подтверждение того же механизма. Государство — тоже граф связей. ОПЖ — ζ-потолок. Возрастная структура — ζ-эффективный. Страны с ОПЖ 73–75 и молодой популяцией (Сирия, FSI = 108) — каскад не компенсирует. Страны с ОПЖ 74 и старой популяцией (Mauritius, FSI = 38) — компенсирует. Одинаковый потолок, разный $\zeta_{\text{эфф}}$.

На заводе — то же самое. Два завода с одинаковым оборудованием и одинаковыми вероятностями отказов, но разной квалификацией операторов (разный β) — дают радикально разный простой.

### 9.3. Ограничения

1. **β задаётся экспертно**, не вычисляется из данных. Это оценка квалификации, регламента, буфера. Возможна калибровка по историческим данным: если линия останавливалась чаще, чем предсказывает ГПСЧ — β был выше номинального.

2. **ζ-насыщение линейно** — упрощение. Реальная зависимость связей от наработки может быть нелинейной (S-кривая, ступенька). Линейная модель — первое приближение.

3. **49 — структурная константа каскада**, не результат оптимизации. Её нельзя «подобрать» под конкретный завод. Можно только соблюдать.

### 9.4. Область применения


|Объект|Узлы|Связи|ζ-аналог|
|:---|:---|:---|:---|
|Производственная линия|Станки|Технологические|Наработка до капремонта|
|Энергосистема|Подстанции|Линии передач|Время эксплуатации|
|Логистическая цепь|Склады|Маршруты|Время работы контракта|
|IT-инфраструктура|Серверы|Сетевые связи|Время с обновления|
|Медицинская система|Отделения|Направления|ОПЖ пациента|
|Государство|Институты|Регуляторные|ОПЖ населения|

Каскадный слой применим везде, где есть граф узлов с связями и возмущения, которые распространяются.

## 10. Выводы

1. **ГПСЧ моделирует узлы, каскад — связи.** Промышленные модели без каскадного слоя слепы к распространению отказов. Они правильно считают частоту, но не видят каскад.

2. **β — главный параметр.** Коэффициент передачи связи определяет, погаснет отказ или снесёт линию. β зависит от квалификации, регламента, буфера — не от надёжности станка.

3. **49 — потолок, не рекомендация.** Превышение ζ-потолка превращает связи в шум. Модель без каскада рекомендует избыточные связи; каскадная модель — качество и соблюдение предела.

4. **Наработка = ζ-насыщение.** Молодой станок — мало связей, слабая компенсация. Зрелый — 49 связей, полная компенсация. Аналог ОПЖ на масштабе оборудования.

5. **Паника разрушает при любом оборудовании.** β \> 1 — каскад усиливается, простой растёт в 8 раз. Качество связей важнее надёжности узлов.

6. **Реализация — 50 строк поверх существующей модели.** Не нужно переписывать. Нужно добавить граф связей, `propagate()` и ζ-поправку.

## Литература

\[1\] Шунько М.Г. Трёхтактный пермутационный цикл дискретного отображения с фикс-поинтом $b^* = 4/9$. Engee, 2026.

\[2\] Шунько М.Г. Каскадная развёртка из неустойчивого истока: четыре фазы, дискретный спектр и условия стабильности кластеров. Engee, 2026.

\[3\] Шунько М.Г. Каскад, ζ-порог и продолжительность жизни: от масштаба наблюдателя до хрупкости государств. 2026.

\[4\] Шунько М.Г. Шаг 50: орбита Земли, число Авогадро и граница стабильности. Engee, 2026.

\[5\] Шунько М.Г. perfect\_random — каскадный генератор псевдослучайных чисел. GitHub, 2026.